# 🔗 Workflow Integration - Complete SAR Processing System

Welcome to Phase 4 of the Financial Services Agentic AI Project!

In this notebook, you'll integrate both AI agents into a complete **end-to-end SAR processing workflow** that demonstrates real-world financial compliance automation.

## 🎯 Learning Objectives
- Build a complete two-stage AI workflow with human oversight
- Implement human-in-the-loop decision gates for compliance
- Generate complete SAR documents from AI analysis
- Create comprehensive audit trails for regulatory examination
- Demonstrate cost optimization through intelligent agent coordination

## 📋 Business Context
This workflow simulates how banks actually process suspicious activity reports:
1. **Risk Screening**: AI agents analyze transaction patterns for suspicious activity
2. **Human Review**: Compliance officers review AI findings before proceeding
3. **Narrative Generation**: Only approved cases get full compliance documentation
4. **SAR Filing**: Complete regulatory forms are generated for submission
5. **Audit Documentation**: Every decision is logged for regulatory examination

## 🏗️ System Architecture

```
📊 CSV Data → 🔍 Risk Analyst → 👤 Human Decision → ✅ Compliance Officer → 📄 SAR Document
              (Chain-of-Thought)    (Gate)         (ReACT Framework)     (FinCEN Ready)
```

## 🚀 Prerequisites Check

Before starting, ensure you have completed:
- ✅ Phase 1: Foundation components (`foundation_sar.py`)
- ✅ Phase 2: Risk Analyst Agent (`risk_analyst_agent.py`)
- ✅ Phase 3: Compliance Officer Agent (`compliance_officer_agent.py`)
- ✅ Both agents pass their comprehensive test scenarios

If any component is missing, return to previous notebooks to complete implementation.

In [1]:
# Setup and Environment Configuration
import os
import sys
import json
import pandas as pd
import uuid
import hashlib
from datetime import datetime, timedelta
from typing import Any, Dict, List, Set, Tuple
from dotenv import load_dotenv

# Add src directory to Python path for module imports
sys.path.append(os.path.abspath('../src'))

# Load environment variables
load_dotenv('../.env')

print("📚 Libraries imported successfully!")
print("🔐 Environment variables loaded")
print("📂 Source directory added to Python path")

📚 Libraries imported successfully!
🔐 Environment variables loaded
📂 Source directory added to Python path


In [2]:
# OpenAI Setup for Vocareum
import openai

# Initialize OpenAI client for Vocareum
openai_api_key = os.getenv('OPENAI_API_KEY')

if not openai_api_key:
    print("⚠️ WARNING: No OpenAI API key found!")
    print("Please set OPENAI_API_KEY in your .env file")
    print("Get your Vocareum OpenAI API key from 'Cloud Resources' in your workspace")
else:
    # Vocareum requires routing through their servers
    client = openai.OpenAI(
        base_url="https://openai.vocareum.com/v1",
        api_key=openai_api_key
    )
    print("✅ OpenAI client initialized with Vocareum routing")
    print("📍 Base URL: https://openai.vocareum.com/v1")

✅ OpenAI client initialized with Vocareum routing
📍 Base URL: https://openai.vocareum.com/v1


In [3]:
# Import implemented components and create agent instances
from foundation_sar import (
    CustomerData,
    AccountData,
    TransactionData,
    CaseData,
    RiskAnalystOutput,
    ComplianceOfficerOutput,
    ExplainabilityLogger,
    DataLoader,
    load_csv_data
)
from risk_analyst_agent import RiskAnalystAgent
from compliance_officer_agent import ComplianceOfficerAgent

os.makedirs("../outputs/audit_logs", exist_ok=True)
explainability_logger = ExplainabilityLogger("../outputs/audit_logs/workflow_integration.jsonl")
data_loader = DataLoader(explainability_logger)
risk_agent = RiskAnalystAgent(client, explainability_logger)
compliance_agent = ComplianceOfficerAgent(client, explainability_logger)

print("✅ Components imported and agents initialized")

✅ Components imported and agents initialized


## 📊 Step 1: Data Loading and Preprocessing

Load the financial data and prepare it for analysis.

In [4]:
# Load and preprocess financial data

def load_and_preprocess_data() -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Load the CSVs, clean optional fields, and return lists of dicts for DataLoader."""
    print("📊 Loading Financial Data")
    try:
        # ssn_last_4 must stay a string (keeps leading zeros, matches the schema)
        customers_df = pd.read_csv("../data/customers.csv", dtype={'ssn_last_4': str})
        accounts_df = pd.read_csv("../data/accounts.csv")
        transactions_df = pd.read_csv("../data/transactions.csv")

        # Handle NaN values in optional fields
        transactions_df['counterparty'] = transactions_df['counterparty'].fillna('')
        transactions_df['location'] = transactions_df['location'].fillna('')
        customers_df['phone'] = customers_df['phone'].fillna('')

        customers_data = customers_df.to_dict('records')
        accounts_data = accounts_df.to_dict('records')
        transactions_data = transactions_df.to_dict('records')

        print(f"📈 Loaded: {len(customers_data)} customers, {len(accounts_data)} accounts, {len(transactions_data)} transactions")
        return customers_data, accounts_data, transactions_data
    except Exception as e:
        print(f"❌ Failed to load data: {type(e).__name__}: {e}")
        raise

# Load data
customers_data, accounts_data, transactions_data = load_and_preprocess_data()

📊 Loading Financial Data
📈 Loaded: 150 customers, 178 accounts, 4268 transactions


## 🎯 Step 2: Customer Risk Screening

Implement intelligent customer screening to identify high-risk cases for detailed analysis.

In [5]:
# Risk-based customer screening

def screen_high_risk_customers(customers_data: List[Dict[str, Any]], accounts_data: List[Dict[str, Any]],
                               transactions_data: List[Dict[str, Any]], top_n: int = 5) -> List[Dict[str, Any]]:
    """
    Screen customers and return the highest-risk ones for detailed analysis.

    Screening criteria:
    1. High risk ratings (Medium, High)
    2. Large transaction amounts (>$100K total)
    3. High transaction frequency (>50 transactions)
    4. Recent activity patterns
    
    A customer is selected when at least two criteria are met (recent-activity patterns
    are left to the Risk Analyst agent). Returns the top N, ranked by number of risk
    flags and then total transaction amount.
    """
    print("🔍 Customer Risk Screening")
    
    selected_customers = []
    for customer in customers_data:
        # Get customer accounts and transactions
        customer_accounts = [acc for acc in accounts_data if acc['customer_id'] == customer['customer_id']]
        customer_transactions = [txn for txn in transactions_data if any(txn['account_id'] == acc['account_id'] for acc in customer_accounts)]
        
        # Calculate risk indicators
        total_amount = sum(abs(txn['amount']) for txn in customer_transactions)
        transaction_count = len(customer_transactions)
        risk_rating = customer['risk_rating']
        
        # Apply screening criteria
        risk_flags = []
        if risk_rating in ['Medium', 'High']:
            risk_flags.append('high_risk_rating')
        if total_amount > 100000:
            risk_flags.append('large_amounts')
        if transaction_count > 50:
            risk_flags.append('high_frequency')
        
        # Select high-risk customers
        if len(risk_flags) >= 2:  # Multiple risk flags
            selected_customers.append({
                'customer': customer,
                'accounts': customer_accounts,
                'transactions': customer_transactions,
                'total_amount': total_amount,
                'transaction_count': transaction_count,
                'risk_flags': risk_flags
            })
    
    # Sort by risk score and take top N
    selected_customers.sort(key=lambda x: (len(x['risk_flags']), x['total_amount']), reverse=True)
    return selected_customers[:top_n]

# Run customer screening
selected_customers = screen_high_risk_customers(customers_data, accounts_data, transactions_data)
print (f'''There are {len(selected_customers)} selected customers''')

🔍 Customer Risk Screening
There are 5 selected customers


## 🤖 Step 3: Two-Stage AI Analysis with Human Gates

Implement the core two-stage workflow:
1. **Stage 1**: Risk Analyst performs Chain-of-Thought analysis
2. **Human Gate**: Review and decision to proceed
3. **Stage 2**: Compliance Officer generates ReACT narratives (only if approved)

In [13]:
def run_two_stage_sar_workflow(selected_customers: List[Dict[str, Any]]) -> Tuple[list, List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Complete two-stage SAR processing workflow
    
    For each customer:
    1. Create CaseData object
    2. Run Risk Analyst analysis (Chain-of-Thought)
    3. Present findings to human reviewer
    4. Get human decision (proceed/reject)
    5. If approved: Run Compliance Officer (ReACT)
    6. Generate complete SAR document
    7. Log all decisions for audit
    """
    print("🤖 Two-Stage SAR Processing Workflow")
    
    # Initialize tracking
    processed_cases = []
    approved_sars = []
    rejected_cases = []
    audit_decisions = []
    
    for i, customer_data in enumerate(selected_customers, 1):
        print(f"\n🔍 CUSTOMER {i}/{len(selected_customers)}: {customer_data['customer']['name']}")
        print("=" * 60)
        
        try:
            # Create case data
            case_data = data_loader.create_case_from_data(
                customer_data['customer'],
                customer_data['accounts'], 
                customer_data['transactions']
            )
            
            # STAGE 1: Risk Analysis
            print("🔍 STAGE 1: Risk Analysis")
            risk_analysis = risk_agent.analyze_case(case_data)
            processed_cases.append(case_data)
            
            # Display analysis results
            print(f"Classification: {risk_analysis.classification}")
            print(f"Confidence: {risk_analysis.confidence_score:.2f}")
            print(f"Risk Level: {risk_analysis.risk_level}")
            print(f"Reasoning: {risk_analysis.reasoning}")
            
            # HUMAN DECISION GATE
            decision = input("🤔 Proceed with SAR filing? (yes/no): ").strip().lower()
            should_proceed = decision in ['yes', 'y']
            
            if should_proceed:
                # STAGE 2: Compliance Narrative
                print("📝 STAGE 2: Compliance Narrative Generation")
                compliance_review = compliance_agent.generate_compliance_narrative(case_data, risk_analysis)
                
                # Generate complete SAR document
                sar_document = create_sar_document(case_data, risk_analysis, compliance_review)
                
                # Save SAR
                save_sar_document(sar_document)
                approved_sars.append(sar_document)
                
                print(f"✅ SAR FILED: {sar_document['sar_metadata']['sar_id']}")
            else:
                rejected_cases.append({'case_id': case_data.case_id, 'reason': 'human_rejection'})
                print("❌ SAR REJECTED by human reviewer")
            
            # Log decision
            audit_decisions.append({
                'case_id': case_data.case_id,
                'customer_name': case_data.customer.name,
                'decision': 'PROCEED' if should_proceed else 'REJECT',
                'ai_classification': risk_analysis.classification,
                'ai_confidence': risk_analysis.confidence_score,
                'reviewer_decision': decision
            })
            
        except Exception as e:
            print(f"❌ Error processing customer: {e}")
    
    return processed_cases, approved_sars, rejected_cases, audit_decisions

# Run the complete workflow
processed_cases, approved_sars, rejected_cases, audit_decisions = run_two_stage_sar_workflow(selected_customers)

🤖 Two-Stage SAR Processing Workflow

🔍 CUSTOMER 1/5: Jacqueline Rodriguez
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.91
Risk Level: High
Reasoning: Multiple high-value outgoing wire transfers shortly after a large incoming transfer suggest layering in money laundering. The amounts are inconsistent with typical business activity and lack clear legitimate purpose.
📝 STAGE 2: Compliance Narrative Generation
📄 Creating SAR Document
✅ SAR FILED: SAR_7260d646-32d1-429d-a690-d33ece6983ec

🔍 CUSTOMER 2/5: Michael Stanley
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.92
Risk Level: Critical
Reasoning: Multiple large outgoing wire transfers shortly after a significant incoming wire transfer suggest layering, a common money laundering technique. The amounts and timing are consistent with money laundering typologies.
📝 STAGE 2: Compliance Narrative Generation
📄 Creating SAR Document
✅ SAR FILED: SAR_6566ac5b-306f-4a4e-a010-73c607b5255d

🔍 CUST

## 📄 Step 4: SAR Document Generation

Create complete, FinCEN-ready SAR documents with all required metadata.

In [12]:
# SAR document generation: complete, FinCEN-style documents for regulatory submission

def create_sar_document(case_data: CaseData, risk_analysis: RiskAnalystOutput,
                        compliance_review: ComplianceOfficerOutput) -> Dict[str, Any]:
    """
    Create complete SAR document
    
    SAR document should include:
    1. SAR metadata (ID, filing date, type, checksum)
    2. Subject information (customer details)
    3. Suspicious activity description
    4. AI analysis results
    5. Compliance narrative
    6. Regulatory citations
    7. Filing institution information
    """
    print("📄 Creating SAR Document")
    
    sar_id = f"SAR_{uuid.uuid4()}"
    filing_date = datetime.now().isoformat()

    txns = case_data.transactions
    txn_dates = [t.transaction_date for t in txns]
    transaction_summary = {
        'transaction_count': len(txns),
        'total_amount': round(sum(abs(t.amount) for t in txns), 2),
        'date_range': {
            'start': min(txn_dates).isoformat(),
            'end': max(txn_dates).isoformat()
        },
        'transaction_ids': [t.transaction_id for t in txns]
    }

    sar_document = {
        'sar_metadata': {
            'sar_id': sar_id,
            'filing_date': filing_date,
            'filing_type': 'Suspicious Activity Report',
            'ai_generated': True,
            'review_status': 'human_approved'
        },
        'filing_institution': {
            # Placeholder values (learning project)
            'name': 'Example National Bank',
            'rssd_id': '0000000',
            'ein': '00-0000000',
            'address': '123 Placeholder St, Anytown, USA 00000',
            'contact_office': 'BSA/AML Compliance Department',
            'contact_phone': '555-000-0000'
        },
        'subject_information': {
            'customer_name': case_data.customer.name,
            'customer_id': case_data.customer.customer_id,
            'address': case_data.customer.address,
            'customer_since': case_data.customer.customer_since.isoformat(),
            'risk_rating': case_data.customer.risk_rating
        },
        'suspicious_activity': {
            'classification': risk_analysis.classification,
            'risk_level': risk_analysis.risk_level,
            'confidence_score': risk_analysis.confidence_score,
            'narrative': compliance_review.narrative,
            'key_indicators': risk_analysis.key_indicators,
            'ai_reasoning': risk_analysis.reasoning,
            'transaction_summary': transaction_summary
        },
        'regulatory_compliance': {
            'citations': getattr(compliance_review, 'regulatory_citations', []),
            'narrative_word_count': len(compliance_review.narrative.split()),
            'compliance_status': 'approved'
        },
        'audit_trail': {
            'case_id': case_data.case_id,
            'processing_date': filing_date,
            'ai_agents_used': ['RiskAnalyst', 'ComplianceOfficer'],
            'human_reviewer': 'compliance_officer'
        }
    }

    # Checksum over canonical JSON of the document (computed before the checksum field exists)
    canonical = json.dumps(sar_document, sort_keys=True)
    sar_document['sar_metadata']['checksum'] = hashlib.sha256(canonical.encode('utf-8')).hexdigest()

    return sar_document


def save_sar_document(sar_document: Dict[str, Any]) -> None:
    """Save SAR document to outputs directory"""
    os.makedirs("../outputs/filed_sars", exist_ok=True)
    filename = f"../outputs/filed_sars/{sar_document['sar_metadata']['sar_id']}.json"
    with open(filename, 'w') as f:
        json.dump(sar_document, f, indent=2)

print("📄 SAR document generation functions defined")

📄 SAR document generation functions defined


## 📊 Step 5: Workflow Metrics and Analysis

Analyze the efficiency and effectiveness of your AI-powered SAR processing system.

In [ ]:
# Workflow metrics and AI-vs-human coherence analysis

def analyze_workflow_efficiency(processed_cases: list, approved_sars: List[Dict[str, Any]],
                                rejected_cases: List[Dict[str, Any]], audit_decisions: List[Dict[str, Any]]) -> None:
    """
    Print workflow metrics: cases processed, SARs filed, approval/rejection rates and
    the cost saving from only running the compliance stage on approved cases.
    """
    print("📊 Workflow Efficiency Analysis")

    total_cases = len(processed_cases)
    approved_cases = len(approved_sars)
    rejected_cases_count = len(rejected_cases)

    if total_cases > 0:
        approval_rate = approved_cases / total_cases
        rejection_rate = rejected_cases_count / total_cases
    else:
        approval_rate = rejection_rate = 0

    print(f"📈 WORKFLOW METRICS:")
    print(f"   Total Cases Processed: {total_cases}")
    print(f"   SARs Filed: {approved_cases}")
    print(f"   Cases Rejected: {rejected_cases_count}")
    print(f"   Approval Rate: {approval_rate:.1%}")
    print(f"   Rejection Rate: {rejection_rate:.1%}")

    # Cost optimization analysis
    print(f"\n💰 COST OPTIMIZATION:")
    print(f"   Two-stage processing saves costs by only running")
    print(f"   expensive compliance generation on approved cases")
    print(f"   Cost savings: {rejection_rate:.1%} of compliance calls avoided")

def validate_ai_decisions(audit_decisions: List[Dict[str, Any]]) -> None:
    """
    Measure coherence between the AI's call and the human reviewer's decision.

    This is NOT an accuracy measure: there are no ground-truth labels, and the reviewer
    sees the AI's reasoning before deciding. It shows where the two agree or diverge.

    AI flagged   = classification is anything other than 'Other'
    Human files  = decision is PROCEED
    Agreement    = AI flagged and human files, or AI did not flag and human declines
    """
    print("\n📋 AI vs HUMAN COHERENCE")

    if not audit_decisions:
        print("   No decisions to compare")
        return

    def ai_flagged(d):
        return d['ai_classification'] != 'Other'

    def human_files(d):
        return d['decision'] == 'PROCEED'

    agree = [d for d in audit_decisions if ai_flagged(d) == human_files(d)]
    # Divergence type A: AI flagged, human declined
    flagged_declined = [d for d in audit_decisions if ai_flagged(d) and not human_files(d)]
    # Divergence type B: AI did not flag, human filed
    cleared_filed = [d for d in audit_decisions if not ai_flagged(d) and human_files(d)]

    ai_flagged_n = sum(ai_flagged(d) for d in audit_decisions)
    ai_cleared_n = len(audit_decisions) - ai_flagged_n

    def pct(n, d):
        return f"{n / d:.1%}" if d else "n/a"

    print(f"   Cases compared: {len(audit_decisions)}")
    print(f"   Agreement: {len(agree)}/{len(audit_decisions)} ({pct(len(agree), len(audit_decisions))})")
    print(f"   AI flagged, human declined: {len(flagged_declined)}  ({pct(len(flagged_declined), ai_flagged_n)} of AI-flagged cases)")
    print(f"   AI cleared, human filed:    {len(cleared_filed)}  ({pct(len(cleared_filed), ai_cleared_n)} of AI-cleared cases)")

    # Does AI confidence track agreement with the human?
    def mean_conf(cases):
        return f"{sum(d['ai_confidence'] for d in cases) / len(cases):.2f}" if cases else "n/a"

    disagree = flagged_declined + cleared_filed
    print(f"   Mean AI confidence when aligned: {mean_conf(agree)}   when diverging: {mean_conf(disagree)}")

    for label, cases in (("AI flagged, human declined", flagged_declined),
                         ("AI cleared, human filed", cleared_filed)):
        for d in cases:
            print(f"     - {label}: {d['customer_name']} ({d['ai_classification']}, confidence {d['ai_confidence']:.2f})")

# Run analysis
analyze_workflow_efficiency(processed_cases, approved_sars, rejected_cases, audit_decisions)
validate_ai_decisions(audit_decisions)

## 🏁 Step 6: Complete System Demonstration

Test your complete system with comprehensive scenarios to validate production readiness.

In [9]:
# Complete System Demonstration
# Runs the full workflow on high-risk AND lower-risk customers so the demo shows
# a spread of AI confidence levels and both approval and rejection decisions.

def select_contrast_customers(customers_data: List[Dict[str, Any]], accounts_data: List[Dict[str, Any]],
                              transactions_data: List[Dict[str, Any]], exclude_ids: Set[str],
                              n: int = 2, min_txns: int = 5) -> List[Dict[str, Any]]:
    """Pick n lower-volume customers (not already selected) to contrast with the high-risk ones."""
    candidates = []
    for customer in customers_data:
        if customer['customer_id'] in exclude_ids:
            continue
        customer_accounts = [acc for acc in accounts_data if acc['customer_id'] == customer['customer_id']]
        account_ids = {acc['account_id'] for acc in customer_accounts}
        customer_transactions = [txn for txn in transactions_data if txn['account_id'] in account_ids]
        if len(customer_transactions) < min_txns:
            continue
        total_amount = sum(abs(txn['amount']) for txn in customer_transactions)
        candidates.append({
            'customer': customer,
            'accounts': customer_accounts,
            'transactions': customer_transactions,
            'total_amount': total_amount,
            'transaction_count': len(customer_transactions),
            'risk_flags': []
        })
    candidates.sort(key=lambda x: x['total_amount'])
    return candidates[:n]

def demonstrate_complete_system() -> Tuple[list, List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Run the complete system demonstration:
    1. Screen the highest-risk customers (top 3)
    2. Add 2 lower-risk customers for contrast
    3. Run the two-stage workflow (you act as the human reviewer)
    4. Show efficiency metrics
    """
    print("🏁 Complete SAR Processing System Demonstration")

    customers_data, accounts_data, transactions_data = load_and_preprocess_data()

    high_risk = screen_high_risk_customers(customers_data, accounts_data, transactions_data, top_n=3)
    contrast = select_contrast_customers(
        customers_data, accounts_data, transactions_data,
        exclude_ids={c['customer']['customer_id'] for c in high_risk}
    )
    selected = high_risk + contrast
    print(f"📋 Demo set: {len(high_risk)} high-risk + {len(contrast)} lower-risk customers")

    processed, approved, rejected, audit = run_two_stage_sar_workflow(selected)

    analyze_workflow_efficiency(processed, approved, rejected, audit)

    print("🎉 System demonstration complete!")
    print("📄 SAR documents saved to: ../outputs/filed_sars/")
    print("📊 Audit logs saved to: ../outputs/audit_logs/")
    return processed, approved, rejected, audit

demo_processed, demo_approved, demo_rejected, demo_audit = demonstrate_complete_system()


🏁 Complete SAR Processing System Demonstration
📊 Loading Financial Data
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Customer Risk Screening
📋 Demo set: 3 high-risk + 2 lower-risk customers
🤖 Two-Stage SAR Processing Workflow

🔍 CUSTOMER 1/5: Jacqueline Rodriguez
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.92
Risk Level: High
Reasoning: Large incoming wire followed by multiple large outgoing wires to different entities suggests layering, a common money laundering technique. The amounts and frequency are inconsistent with typical business activity.
📝 STAGE 2: Compliance Narrative Generation
📄 Creating SAR Document
✅ SAR FILED: SAR_e42f6501-1058-41b5-aac7-b52aa2ced393

🔍 CUSTOMER 2/5: Michael Stanley
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.92
Risk Level: Critical
Reasoning: Multiple large outgoing wire transfers shortly after a significant incoming wire transfer suggest layering in money laundering. The amounts and

## 📝 Implementation Checklist

### ✅ Workflow Integration Deliverables
- [x] **Data Loading**: Load and preprocess CSV data with proper error handling
- [x] **Customer Screening**: Implement risk-based screening to identify high-risk cases
- [x] **Two-Stage Workflow**: Build complete Risk Analyst → Human Gate → Compliance Officer flow
- [x] **Human Decision Gates**: Implement interactive approval/rejection points
- [x] **SAR Document Generation**: Create complete FinCEN-ready documents with metadata
- [x] **Audit Trail Creation**: Log all decisions and reasoning for regulatory examination
- [ ] **Efficiency Metrics**: Calculate cost optimization and processing efficiency
- [x] **System Demonstration**: Test complete workflow with multiple scenarios

### ✅ Testing and Validation Requirements
- [x] **Component Validation**: Verify all foundation components and agents are available
- [x] **Integration Testing**: Run comprehensive test suites for all components with proper sys.path setup
- [x] **End-to-End Testing**: Test complete workflow with automated scenarios
- [x] **Error Handling Testing**: Validate graceful handling of edge cases and failures
- [x] **Output Validation**: Ensure SAR documents meet regulatory standards
- [ ] **Performance Testing**: Measure workflow efficiency and processing times

### ✅ Technical Requirements
- [x] **Error Handling**: Robust exception handling for all workflow steps
- [x] **Data Validation**: Proper validation of all inputs and outputs
- [x] **File Management**: Organize outputs in appropriate directories
- [x] **Logging**: Comprehensive audit logging for compliance
- [x] **Performance**: Efficient processing of multiple cases
- [x] **User Experience**: Clear prompts and feedback for human reviewers
- [x] **Test Infrastructure**: Proper test imports and sys.path configuration

### ✅ Business Requirements  
- [x] **Regulatory Compliance**: Ensure all SAR documents meet FinCEN requirements
- [x] **Cost Optimization**: Demonstrate savings from two-stage processing
- [x] **Audit Readiness**: Create examination-ready documentation
- [x] **Quality Assurance**: Validate AI decisions with human oversight
- [x] **Scalability**: Design for processing larger datasets
- [x] **Production Readiness**: Complete testing validates system reliability

## 🎯 Success Criteria

By completion, your integrated system should:
- ✅ Process real financial data with proper validation
- ✅ Execute complete two-stage AI workflow with human gates
- ✅ Generate regulatory-compliant SAR documents
- ✅ Create comprehensive audit trails for all decisions
- ✅ Demonstrate measurable cost optimization benefits
- ✅ Handle errors gracefully and provide clear user feedback
- ✅ Pass all integration and end-to-end tests
- ✅ Meet production-ready quality standards

## 🚀 Next Steps

1. **Complete Implementation**: Make sure every section has working code
2. **Run Integration Tests**: Validate all components work together properly
3. **Execute End-to-End Tests**: Test complete workflow with automated scenarios
4. **Test Thoroughly**: Run complete workflow with various manual scenarios
5. **Validate Outputs**: Ensure SAR documents meet regulatory requirements
6. **Document Results**: Create final project documentation and metrics
7. **Prepare Presentation**: Demonstrate your system's capabilities and business value

**Congratulations on building a complete AI-powered SAR processing system! 🎉**

## 🧪 Step 7: Workflow Testing and Validation

Before finalizing your implementation, validate your complete system with comprehensive testing.

In [ ]:
# 🧪 Workflow Integration Testing
# Validate your complete system with integration tests

import sys
import os

# Add tests directory to Python path for importing test modules
project_root = os.path.abspath('..')
tests_path = os.path.join(project_root, 'tests')
if tests_path not in sys.path:
    sys.path.insert(0, tests_path)
# The test modules import `from src.xxx`, so the project root must be importable too
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"📁 Added tests directory to Python path: {tests_path}")

def run_integration_tests() -> bool:
    """
    Run comprehensive integration tests to validate the complete workflow
    
    Tests include:
    1. Foundation components integration
    2. Agent communication and data flow
    3. End-to-end workflow execution
    4. Error handling and edge cases
    5. Output validation and compliance
    """
    print("🧪 Comprehensive Integration Testing")
    
    try:
        # Import all test modules
        from test_foundation import TestCustomerData, TestAccountData, TestTransactionData, TestCaseData
        from test_risk_analyst import TestRiskAnalystAgent
        from test_compliance_officer import TestComplianceOfficerAgent
        import pytest
        
        print("🔍 Running Foundation Component Tests...")
        foundation_result = pytest.main([
            f"{tests_path}/test_foundation.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("🔍 Running Risk Analyst Agent Tests...")
        risk_result = pytest.main([
            f"{tests_path}/test_risk_analyst.py", 
            "-v", 
            "--tb=short"
        ])
        
        print("📝 Running Compliance Officer Agent Tests...")
        compliance_result = pytest.main([
            f"{tests_path}/test_compliance_officer.py", 
            "-v", 
            "--tb=short"
        ])
        
        # Calculate overall test results
        all_passed = foundation_result == 0 and risk_result == 0 and compliance_result == 0
        
        print("\n" + "="*60)
        print("📊 INTEGRATION TEST RESULTS:")
        print(f"   Foundation Components: {'✅ PASS' if foundation_result == 0 else '❌ FAIL'}")
        print(f"   Risk Analyst Agent: {'✅ PASS' if risk_result == 0 else '❌ FAIL'}")
        print(f"   Compliance Officer Agent: {'✅ PASS' if compliance_result == 0 else '❌ FAIL'}")
        print(f"   Overall Status: {'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED'}")
        
        if all_passed:
            print("\n🎉 Your system is ready for production workflow testing!")
            print("📝 Proceed to run the complete system demonstration.")
        else:
            print("\n⚠️ Fix failing tests before running the complete workflow.")
            print("📝 Return to previous notebooks to fix component issues.")
        
        return all_passed
            
    except ImportError as e:
        print(f"❌ Import Error: {e}")
        print("💡 Make sure all components are implemented:")
        print("   • foundation_sar.py")
        print("   • risk_analyst_agent.py") 
        print("   • compliance_officer_agent.py")
        return False

def validate_workflow_components() -> bool:
    """Validate that all required components are available for integration"""
    print("🔍 Validating Workflow Components")
    
    components_status = {
        'foundation_sar': False,
        'risk_analyst_agent': False,
        'compliance_officer_agent': False,
        'test_modules': False
    }
    
    try:
        # Check foundation components
        from foundation_sar import CustomerData, CaseData, ExplainabilityLogger, DataLoader
        components_status['foundation_sar'] = True
        print("✅ Foundation components available")
    except ImportError:
        print("❌ Foundation components not available")
    
    try:
        # Check risk analyst agent
        from risk_analyst_agent import RiskAnalystAgent
        components_status['risk_analyst_agent'] = True
        print("✅ Risk Analyst Agent available")
    except ImportError:
        print("❌ Risk Analyst Agent not available")
    
    try:
        # Check compliance officer agent
        from compliance_officer_agent import ComplianceOfficerAgent
        components_status['compliance_officer_agent'] = True
        print("✅ Compliance Officer Agent available")
    except ImportError:
        print("❌ Compliance Officer Agent not available")
    
    try:
        # Check test modules
        from test_foundation import TestCustomerData
        from test_risk_analyst import TestRiskAnalystAgent  
        from test_compliance_officer import TestComplianceOfficerAgent
        components_status['test_modules'] = True
        print("✅ Test modules available")
    except ImportError:
        print("❌ Test modules not available")
    
    all_ready = all(components_status.values())
    
    print(f"\n📊 Component Status: {'✅ ALL READY' if all_ready else '⚠️ INCOMPLETE'}")
    if not all_ready:
        print("💡 Complete missing components before running integration tests")
    
    return all_ready

# Run component validation
components_ready = validate_workflow_components()

# Run integration tests if components are ready
if components_ready:
    print("\n🚀 All components ready - you can run integration tests!")
    run_integration_tests()
else:
    print("\n📋 Complete component implementation first, then run integration tests")

In [11]:
# 🎯 End-to-End Workflow Testing
# Runs the real workflow with the human gate's input() patched to scripted answers

from unittest.mock import patch

def test_complete_workflow() -> bool:
    """
    End-to-end test of run_two_stage_sar_workflow.

    - Screens 2 customers and runs the real two-stage workflow (live API calls)
    - Patches input() so the human gate answers 'yes' then 'no'
    - Checks outputs: approved/rejected counts, SAR file + checksum, audit trail
    """
    print("🎯 End-to-End Workflow Testing")
    try:
        customers, accounts, transactions = load_and_preprocess_data()
        selected = screen_high_risk_customers(customers, accounts, transactions, top_n=2)
        assert len(selected) == 2, f"Expected 2 screened customers, got {len(selected)}"

        # Scripted human decisions: approve the first case, reject the second
        with patch('builtins.input', side_effect=['yes', 'no']):
            processed, approved, rejected, audit = run_two_stage_sar_workflow(selected)

        # Workflow outputs
        assert len(processed) == 2, f"Expected 2 processed cases, got {len(processed)}"
        assert len(approved) == 1, f"Expected 1 approved SAR, got {len(approved)}"
        assert len(rejected) == 1, f"Expected 1 rejected case, got {len(rejected)}"

        # Approved SAR: file saved and checksum verifies
        sar = approved[0]
        sar_id = sar['sar_metadata']['sar_id']
        path = f"../outputs/filed_sars/{sar_id}.json"
        assert os.path.exists(path), f"SAR file not saved: {path}"
        with open(path) as f:
            saved = json.load(f)
        stored_checksum = saved['sar_metadata'].pop('checksum')
        recomputed = hashlib.sha256(json.dumps(saved, sort_keys=True).encode('utf-8')).hexdigest()
        assert stored_checksum == recomputed, "SAR checksum does not match saved content"

        # Rejected case: no SAR file written
        rejected_id = rejected[0]['case_id']
        assert all(s['audit_trail']['case_id'] != rejected_id for s in approved), "Rejected case produced a SAR"

        # Audit trail covers every processed case with the right decisions
        assert {a['case_id'] for a in audit} == {c.case_id for c in processed}, "Audit trail incomplete"
        assert [a['decision'] for a in audit] == ['PROCEED', 'REJECT'], f"Unexpected decisions: {[a['decision'] for a in audit]}"

        print("\n🎉 END-TO-END TEST PASSED")
        return True

    except AssertionError as e:
        print(f"\n❌ END-TO-END TEST FAILED: {e}")
        return False
    except Exception as e:
        print(f"\n❌ END-TO-END TEST ERRORED: {type(e).__name__}: {e}")
        return False

# Run end-to-end test (makes live API calls)
test_success = test_complete_workflow()


🎯 End-to-End Workflow Testing
📊 Loading Financial Data
📈 Loaded: 150 customers, 178 accounts, 4268 transactions
🔍 Customer Risk Screening
🤖 Two-Stage SAR Processing Workflow

🔍 CUSTOMER 1/2: Jacqueline Rodriguez
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.91
Risk Level: High
Reasoning: Significant incoming wire transfer followed by multiple large outgoing wire transfers to various entities within a short timeframe suggests layering, a common money laundering technique.
📝 STAGE 2: Compliance Narrative Generation
📄 Creating SAR Document
✅ SAR FILED: SAR_04f273bb-6c64-4e94-9f75-cba75c6bf55f

🔍 CUSTOMER 2/2: Michael Stanley
🔍 STAGE 1: Risk Analysis
Classification: Money_Laundering
Confidence: 0.92
Risk Level: High
Reasoning: Large incoming wire transfer followed by rapid, large outgoing wire transfers to multiple entities, inconsistent with typical customer profile, suggests layering stage of money laundering.
❌ SAR REJECTED by human reviewer

🎉 END-TO-END TEST 